# 05 — Anomaly Detection Development

LSTM-Autoencoder pretraining on LogHub HDFS and evaluation.

**Runtime:** Google Colab Pro (T4/L4/A100 GPU recommended)

## Workflow
1. Setup environment and load HDFS data
2. Create model with dynamic `input_dim` from Drain3 templates
3. Pretrain on normal HDFS sequences
4. Plot training curves
5. Calculate threshold on holdout normal data
6. Evaluate on HDFS benchmark (precision/recall/F1)
7. Fine-tune on OTel Demo (placeholder — requires log collection)

In [ ]:
# Cell 1: Setup
# If running on Google Colab, uncomment the following lines:

# from google.colab import drive
# drive.mount('/content/drive')
# !pip install drain3 scikit-learn

import sys
import os

# Add project root to path (adjust if repo is in a different location)
# For Colab: sys.path.insert(0, '/content/drive/MyDrive/opsagent')
# For local: sys.path.insert(0, '..')
sys.path.insert(0, '..')

import numpy as np
import torch
import matplotlib.pyplot as plt

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Cell 2: Load HDFS data
from src.preprocessing.log_parser import LogParser
from src.preprocessing.loghub_preprocessor import LogHubHDFSPreprocessor

HDFS_DATA_PATH = '../data/LogHub/HDFS/'  # Adjust for Colab

# Create shared parser (will be reused for OTel fine-tuning later)
parser = LogParser(persistence_path='../models/drain3/')

# Parse HDFS.log — this takes several minutes for 11M+ lines
preprocessor = LogHubHDFSPreprocessor(
    data_dir=HDFS_DATA_PATH,
    seq_length=10,
    parser=parser,
)
preprocessor.parse()

n_templates = preprocessor.num_templates
print(f'Templates discovered: {n_templates}')
print(f'Blocks parsed: {len(preprocessor._block_sequences)}')

# Get sequences
normal_seqs = preprocessor.get_normal_sequences()
anomalous_seqs = preprocessor.get_anomalous_sequences()
print(f'Normal sequences: {normal_seqs.shape}')
print(f'Anomalous sequences: {anomalous_seqs.shape}')
print(f'Anomaly rate: {len(anomalous_seqs) / (len(normal_seqs) + len(anomalous_seqs)):.4f}')

In [ ]:
# Cell 3: Create model and pretrain
from src.anomaly_detection.pretrain_on_loghub import pretrain_on_hdfs

CHECKPOINT_PATH = '../models/lstm_autoencoder/pretrained_hdfs.pt'

# Pretrain — epochs=50, batch_size=64, lr=0.001, patience=5
model, parser = pretrain_on_hdfs(
    hdfs_data_path=HDFS_DATA_PATH,
    model_save_path=CHECKPOINT_PATH,
    parser=parser,
)

print(f'\nModel parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Checkpoint saved to: {CHECKPOINT_PATH}')

In [ ]:
# Cell 4: Plot training curves
checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu')
history = checkpoint['history']

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(history['train_loss'], label='Train Loss', linewidth=2)
ax.plot(history['val_loss'], label='Val Loss', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('LSTM-AE Pretraining on LogHub HDFS')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../docs/images/hdfs_pretraining_curves.png', dpi=150)
plt.show()

print(f'Final train loss: {history["train_loss"][-1]:.6f}')
print(f'Final val loss:   {history["val_loss"][-1]:.6f}')
print(f'Total epochs:     {len(history["train_loss"])}')

In [ ]:
# Cell 5: Calculate threshold on holdout normal data
from src.anomaly_detection.pretrain_on_loghub import _one_hot_encode
from src.anomaly_detection.threshold import calculate_threshold
from sklearn.model_selection import train_test_split

# Split normal sequences — use the validation portion for threshold
_, holdout_seqs = train_test_split(normal_seqs, test_size=0.2, random_state=42)

# One-hot encode for model input
holdout_data = _one_hot_encode(holdout_seqs, n_templates)
print(f'Holdout sequences for threshold: {holdout_data.shape}')

# Calculate 95th percentile threshold
device = 'cuda' if torch.cuda.is_available() else 'cpu'
threshold = calculate_threshold(
    model, holdout_data, percentile=95, batch_size=256, device=device
)
print(f'\n95th percentile threshold: {threshold:.6f}')

In [ ]:
# Cell 6: HDFS Benchmark — evaluate on labeled data
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

# One-hot encode anomalous sequences
anomalous_data = _one_hot_encode(anomalous_seqs, n_templates)

# Score all sequences
model.eval()
model.to(device)

def get_errors(data, batch_size=256):
    errors = []
    for i in range(0, len(data), batch_size):
        batch = torch.FloatTensor(data[i:i+batch_size]).to(device)
        with torch.no_grad():
            err = model.get_reconstruction_error(batch)
            errors.extend(err.cpu().tolist())
    return np.array(errors)

normal_errors = get_errors(holdout_data)
anomalous_errors = get_errors(anomalous_data)

# Classify using threshold
all_errors = np.concatenate([normal_errors, anomalous_errors])
all_labels = np.concatenate([
    np.zeros(len(normal_errors), dtype=int),
    np.ones(len(anomalous_errors), dtype=int),
])
predictions = (all_errors > threshold).astype(int)

print('HDFS Benchmark Results:')
print(f'  Threshold: {threshold:.6f}')
print(f'  Normal samples:    {len(normal_errors)}')
print(f'  Anomalous samples: {len(anomalous_errors)}')
print()
print(classification_report(all_labels, predictions, target_names=['Normal', 'Anomaly']))

f1 = f1_score(all_labels, predictions)
precision = precision_score(all_labels, predictions)
recall = recall_score(all_labels, predictions)
print(f'F1: {f1:.4f}  Precision: {precision:.4f}  Recall: {recall:.4f}')

In [ ]:
# Cell 7: Error distribution visualization
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(normal_errors, bins=50, alpha=0.7, label='Normal', color='steelblue', density=True)
ax.hist(anomalous_errors, bins=50, alpha=0.7, label='Anomalous', color='coral', density=True)
ax.axvline(threshold, color='red', linestyle='--', linewidth=2, label=f'Threshold (p95={threshold:.4f})')
ax.set_xlabel('Reconstruction Error (MSE)')
ax.set_ylabel('Density')
ax.set_title('LSTM-AE Reconstruction Error Distribution — HDFS')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../docs/images/hdfs_error_distribution.png', dpi=150)
plt.show()

In [ ]:
# Cell 8: Fine-tune on OTel Demo (placeholder)
# This cell will be runnable after Week 6.5 when Promtail is configured
# and a log-enriched baseline has been collected.
#
# from src.anomaly_detection.pretrain_on_loghub import finetune_on_otel_demo
#
# otel_data = {
#     'train': train_features,    # np.ndarray (N, seq_len, feature_dim)
#     'val': val_features,        # np.ndarray (M, seq_len, feature_dim)
#     'input_dim': feature_dim,   # int from FeatureEngineer.feature_dim
# }
#
# finetuned_model = finetune_on_otel_demo(
#     pretrained_model_path='../models/lstm_autoencoder/pretrained_hdfs.pt',
#     otel_data=otel_data,
#     model_save_path='../models/lstm_autoencoder/finetuned_otel.pt',
# )

print('OTel Demo fine-tuning: DEFERRED')
print('Requires: Promtail log collection + 4-8h baseline with logs+metrics')
print('See PROGRESS.md Week 6.5 tasks')